# Giggsdance — 60 fps video with MiniMax H3

Generate a clip with [MiniMax H3](https://huggingface.co/MiniMaxAI/MiniMax-H3), convert it to a correct 60 fps, optionally upscale, and encode a 10-bit master — all on a Modal GPU. Nothing runs on this notebook's machine, so a free Colab runtime is fine.

**Read [NOTICE.md](https://github.com/Hvkki/minimax/blob/main/NOTICE.md) first.** The H3 licence does not grant rights in the **EU, UK, South Korea or USA**, and the restriction covers the model's **outputs**, not just its weights. If you publish a result, mark it as AI-generated.

Run the cells in order. Cell 3 is free and needs no account.

## 1. Install

`!` runs a shell command. Without it the cell is parsed as Python and you get `SyntaxError`.

In [ ]:
!git clone -q https://github.com/Hvkki/minimax.git
%cd minimax
!pip install -q modal numpy pillow
print("installed")

## 2. Modal credentials

Create a token pair at [modal.com/settings/tokens](https://modal.com/settings/tokens) and paste them below. Modal runs your *local* files — there is no GitHub integration to configure.

In [ ]:
import os
os.environ["MODAL_TOKEN_ID"] = ""      # ak-...
os.environ["MODAL_TOKEN_SECRET"] = ""  # as-...

import notebook
notebook.check_setup()

## 3. Free check — no GPU, no account, $0

Runs the real interpolation, geometry and encoding stages on synthetic frames and asserts frame count, fps, bit depth, colour tags and A/V sync. Do this before spending anything.

In [ ]:
notebook.dry_run()

## 4. Optional: measure the cold start

Loading H3 is ~124 GB and happens before a single frame is generated. It is the least predictable cost in the system, so measuring it once makes every later budget reliable.

The **first** run also downloads ~90 GB of weights into a Modal Volume. That happens on cheap CPU and is skipped on every later run.

In [ ]:
notebook.probe()

## 5. Render

Defaults are the cheap ones: 5 s, `native` resolution (**no** super-resolution — it measured at ~84% of post-processing time), 60 fps, 8 steps. `budget_usd` becomes a hard container timeout, so an overrun is killed rather than billed.

In [ ]:
path, report = notebook.render(
    prompt="a paper boat drifting down a rain-filled gutter at night, neon reflections",
    duration_s=5.0,
    resolution="native",   # "1080p" / "1440p" / "2160p" to enable upscaling
    fps=60,
    steps=8,
    budget_usd=1.00,
)
print(path)

## 6. Variations

H3 constraints worth knowing: clips are **5–14.375 s** (frame counts must be `17n+5`), always **24 fps** natively, and the 16:9 canvas is really **1344x768**.

```python
notebook.render(duration_s=10, steps=16)                      # longer, better quality
notebook.render(resolution="1440p", steps=24, budget_usd=3)    # upscaling on
notebook.render(aspect_ratio="9:16", resolution="1080p")       # vertical
notebook.render(seed=42)                                       # reproducible
```

Prompt quality matters a lot. H3's Context-IR module is closed-source, so structured prompts help:

```python
from giggsdance import Storyboard, Shot, DialogueLine
board = Storyboard(
    style="Cinematic 2D vector animation, flat colour",
    shots=[
        Shot(description="A stick figure sprints across a white void.", camera="Static wide"),
        Shot(description="It lands a spinning kick; ripples spread.", start_s=3.0),
    ],
    soundscape="Sharp whooshes, a percussive thud on impact, faint room tone.",
    music="Driving synth arpeggio building to a hit on the kick.",
)
notebook.render(prompt=board.render())
```

## Troubleshooting

| Symptom | Cause |
|---|---|
| `SyntaxError: invalid syntax` on `git clone` | shell command in a Python cell — prefix with `!` |
| `Bad Request: Unsupported URL` | pasting the repo URL into Modal; Modal uploads local files instead |
| `ModuleNotFoundError: giggsdance` | wrong working directory — run `%cd minimax` |
| timeout | raise `budget_usd`, or lower `steps` |
| generation error | `notebook.describe()` dumps the real pipeline signature |

`stages/generate.py` is the one part of this repo never executed by its author — it needs ~124 GB of weights and a large GPU. If something breaks, it is most likely there.